<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 28 — חקר מלא והערכת סוף
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
נחבר את כל החלקים לחקר Titanic: שאלה, נתונים, בחירות בפיתוח, מודל נבחר ותוצאה סופית שאפשר להסביר.

## מטרות
- לקשור שלבי חקר.
- להפריד בחירה מבדיקת סוף.
- לנתח מגבלה.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## המושגים הנחוצים
חקר מלא כולל מקור, שאלה, חלוקה, EDA, הכנה, אימון, אימות, בחירה והערכת סוף. המודל נבחר לפני חשיפת Test. לאחר מכן מתאימים את המתכון לכל Train. טעויות ומגבלות הן חלק מהתוצר. חלוקה אקראית לנוסעים אינה מבטיחה הכללה למשפחות חדשות.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## דוגמה פתורה לפני קוד
לאחר בחירת מועמד באמצעות CV מפיקים פעם אחת תחזיות Test. אפשר לחשב מהן כמה מדדים, אך אין לשנות את המתכון לפי התוצאה ולדווח כאילו לא נחשפה.

</div>

In [ ]:
from sklearn.metrics import f1_score
y_true = [0, 0, 1, 1]
y_pred = [0, 1, 1, 1]
print(f1_score(y_true, y_pred))

<div dir="rtl" lang="he" style="text-align: right;">

**פלט צפוי:** 0.8

## מדוע זה חשוב?
תהליך חקר, תיעוד והערכה עוברים לשתי תכניות ההמשך. מאגר שכבר הכרנו אינו הופך למבחן עצמאי חדש בשנה הבאה.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## יישום מודרך — חלק 1
מקור הקוד: ערכה תהליך החקר הקיימת. בהמשך עוברים לתרגול עצמאי באותו נושא.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

### טעינה ובדיקת קלט
טוענים את מקור הנתונים שהופיע בתרגול. בודקים שדות וממדים לפני שימוש. כתובת Raw שונה מדף התצוגה של GitHub.

</div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,StratifiedKFold,GridSearchCV,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
url="https://raw.githubusercontent.com/arielb30/datasets/main/titanic_data.csv"
raw=pd.read_csv(url)
print("Source shape:",raw.shape)
print("Columns:",raw.columns.tolist())

<div dir="rtl" lang="he" style="text-align: right;">

### תשתית הנתונים והחלוקה
קראו תחילה את היעד ואת שמות המאפיינים. החלוקה שומרת התאמה בין שורות X ל־y. הפרמטר random_state מאפשר שחזור ואינו משפר דיוק בפני עצמו.

</div>

In [ ]:
numeric=['Age', 'Fare']
categorical=['Sex', 'Pclass']
features=numeric+categorical
required=features+["Survived","PassengerId"]
assert set(required).issubset(raw.columns),"Check the supplied schema"
assert raw["PassengerId"].notna().all()
assert raw["PassengerId"].is_unique,"Resolve repeated people before splitting"
assert set(raw["Survived"].dropna().unique()).issubset({0,1})
data=raw.dropna(subset=["Survived"]).copy()
assert data["Age"].dropna().ge(0).all()
assert data["Fare"].dropna().ge(0).all()
X=data[features].copy()
y=data["Survived"].astype(int).copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
assert set(X_train.index).isdisjoint(X_test.index)
print("Eligible / train / test:",len(data),len(X_train),len(X_test))

<div dir="rtl" lang="he" style="text-align: right;">

### חישוב ופירוש
קראו את השמות והיחידות, כתבו ציפייה לפני הרצה ובדקו אותה בפלט. הסבירו מה השתנה ומה נשאר קבוע. אין צורך לשנן פקודה בלי להבין את פעולתה.

</div>

In [ ]:
development=X_train.copy()
development["Survived"]=y_train
development.info()
display(development.describe())
print("Missing:")
print(development.isna().sum())
display(development.groupby("Pclass")["Survived"].agg(["count","sum","mean"]))

<div dir="rtl" lang="he" style="text-align: right;">

### המחשה חזותית
לפני הרצה הגדירו שאלה, סוגי משתנים ויחידות. אחרי הרצה פרשו את הצירים ואת הממצא. אין להסיק סיבתיות או למחוק קצה רק מן הציור.

</div>

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
sns.histplot(data=development,x="Age",bins=12,ax=axes[0])
axes[0].set_title("Training ages (known values)")
axes[0].set_xlabel("Age (years)")
sns.countplot(data=development,x="Survived",ax=axes[1])
axes[1].set_title("Training labels")
axes[1].set_ylabel("Number of records")
plt.tight_layout();plt.show()

<div dir="rtl" lang="he" style="text-align: right;">

### חיפוש מועמדים
קראו את שמות השלבים, רשימת המועמדים ומדד הבחירה. החיפוש מקבל נתוני אימון בלבד; נתוני האימות הפנימיים אינם Test חיצוני.

</div>

In [ ]:
prepare=ColumnTransformer([
 ("numeric",Pipeline([("fill",SimpleImputer(strategy="median")),("scale",StandardScaler())]),numeric),
 ("categorical",Pipeline([("fill",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),categorical)
])
knn=Pipeline([("prepare",prepare),("model",KNeighborsClassifier())])
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
param_grid={"model__n_neighbors":[3,5,9]}
clf=GridSearchCV(knn,param_grid=param_grid,cv=cv,scoring="f1",refit=True,return_train_score=True,error_score="raise")
clf.fit(X_train,y_train)
results=pd.DataFrame(clf.cv_results_)
display(results[["param_model__n_neighbors","mean_train_score","mean_test_score","rank_test_score"]])
print("Best KNN:",clf.best_params_,clf.best_score_)

<div dir="rtl" lang="he" style="text-align: right;">

### אימות בתוך הפיתוח
ההכנה נמצאת בתוך Pipeline. כל קיפול לומד את פרמטריו מתת־האימון שלו. נקרא את ממוצע האימות לפני שנחליט על מודל.

</div>

In [ ]:
svm=Pipeline([("prepare",prepare),("model",SVC(kernel="linear",C=1.0))])
baseline=DummyClassifier(strategy="most_frequent")
svm_scores=cross_val_score(svm,X_train,y_train,cv=cv,scoring="f1",error_score="raise")
baseline_scores=cross_val_score(baseline,X_train,y_train,cv=cv,scoring="f1",error_score="raise")
cv_table=pd.DataFrame({"model":["KNN","Linear SVM","Baseline"],"mean_cv_f1":[clf.best_score_,svm_scores.mean(),baseline_scores.mean()]})
display(cv_table)
# Tie policy: first row in the predefined order wins. Test is not read here.
chosen_name=cv_table.sort_values("mean_cv_f1",ascending=False,kind="stable").iloc[0]["model"]
chosen={"KNN":clf.best_estimator_,"Linear SVM":svm,"Baseline":baseline}[chosen_name]
chosen.fit(X_train,y_train)
baseline.fit(X_train,y_train)
print("Chosen before Test:",chosen_name)

<div dir="rtl" lang="he" style="text-align: right;">

### חישוב ופירוש
קראו את השמות והיחידות, כתבו ציפייה לפני הרצה ובדקו אותה בפלט. הסבירו מה השתנה ומה נשאר קבוע. אין צורך לשנן פקודה בלי להבין את פעולתה.

</div>

In [ ]:
predictions=chosen.predict(X_test)
baseline_predictions=baseline.predict(X_test)
matrix=confusion_matrix(y_test,predictions,labels=[0,1])
display(pd.DataFrame(matrix,index=["Actual 0","Actual 1"],columns=["Predicted 0","Predicted 1"]))
tn,fp,fn,tp=matrix.ravel()
precision_manual=tp/(tp+fp) if tp+fp else 0.0
recall_manual=tp/(tp+fn) if tp+fn else 0.0
f1_manual=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
print("Accuracy:",accuracy_score(y_test,predictions))
print("Precision:",precision_score(y_test,predictions,zero_division=0),precision_manual)
print("Recall:",recall_score(y_test,predictions,zero_division=0),recall_manual)
print("F1:",f1_score(y_test,predictions,zero_division=0),f1_manual)
print("Baseline F1:",f1_score(y_test,baseline_predictions,zero_division=0))
comparison=X_test.copy()
comparison["actual"]=y_test
comparison["predicted"]=predictions
display(comparison[comparison["actual"]!=comparison["predicted"]].head(5))

<div dir="rtl" lang="he" style="text-align: right;">

### חישוב ופירוש
קראו את השמות והיחידות, כתבו ציפייה לפני הרצה ובדקו אותה בפלט. הסבירו מה השתנה ומה נשאר קבוע. אין צורך לשנן פקודה בלי להבין את פעולתה.

</div>

In [ ]:
new_example=pd.DataFrame([{feature:(1 if feature=="SibSp" else 25 if feature=="Age" else 20 if feature=="Fare" else "female" if feature=="Sex" else 3) for feature in features}])
print("Teaching example prediction:",chosen.predict(new_example))

<div dir="rtl" lang="he" style="text-align: right;">

## עצירה לפני המעבדה
האם כמה מדדים מאותה בדיקת סוף הם בעיה?

**התוצר הנדרש:** במחברת החקר המלא משלימים רק מקטעים שטרם בוצעו. אין חובה לפרויקט Titanic נוסף לצד הפרויקט האישי.

</div>